In [1]:
#Note: I've tried running some of this with modin to speed up bits of code, but am getting lots of errors. So going back to pandas.

import pandas as pd
#import modin.pandas as pd

## Loading and Cleaning

We'll start by loading the full data set and converting fips_code into an integer and neighbors into a list of (integer) fips_code values. This will support some merging/functions below.

In [2]:
#Load the ../Data/Merged_Data/eaglei_noaa_era5_full.parquet file
df = pd.read_parquet('../Data/Merged_Data/eaglei_noaa_era5_full.parquet')

In [ ]:
#Drop rows corresponding to years 2022 and newer
#Skipping this for now until after features have been engineered
#df = df[df['YEAR'] < 2022]

In [3]:
#Convert fips_code to a float, round it to the nearest integer, and then convert to an int64
df['fips_code'] = df['fips_code'].astype(float).round().astype('int64')

In [4]:
#Use json to convert each value of neighbors into a list of integers
import json
df['neighbors'] = df['neighbors'].apply(lambda x: json.loads(x))

# NaN Values

Note that the following fips_code values are missing data for percent buried lines:
- 22101: St. Mary Parish, LA; it is adjacent to:
    - Iberia Parish (22045); this has a value of 0.1327997527742176
    - St. Martin Parish (22099); this has a value of 0.1628546644133763
    - Assumption Parish (22007); this has a value of 0.1593958603974588
    - Terrebonne Parish (22109); this has a value of 0.1939858072512728
    - If we use the average of the adjacent counties, it should have a value of 0.162259021. (But see the note below)
- 46102: Oglala Lakota, SD; this code transitioned from fips_code 46113, and should have a value of 0.1205137891106878
- 51720: Norton City, VA; this is in Wise County (fips_code 51195); if we assume that it has the same percentage as the surrounding county, it should have a value of 0.099491631

*Note that each of these missing values should have been taken care of when running the county-level data scripts. Each case is somewhat unique and probably requires its own treatment. In general, it seems problematic to impute these values, so we're not going to write any code that could be used to try to impute them generally.*

The following fips_code values are missing data for Subregion (we can fill these in manually):
- 25001: Barnstable, MA; this has ZIP code 02630 and, according to epa.gov/egrid/power-profiler, it is in the NEWE subregion
- 51131: Northampton, VA; this has 16 ZIP codes, including 23307 which, according to the power-profiler, is in the RFCE subregion

The following fips_code values are missing data for Power_Dependent_Devices_DME_Mean
This should have originally been adjusted in the script that combined the emPower data, and this value propagated through the stages of data cleaning
 For now, we can look this value up from its original fips code:
- 46102: Oglala Lakota, SD; this code transitioned from fips_code 46113

The following fips_code values are missing ERA5 data. With the exception of Hudson, these are all adjacent to the ocean or a large body of water. 
It seems likely that the (rounded) centroid coordinates are not in the ERA5-Land dataset.
We could try to download more comprehensive ERA5 data to look up their values. 
Alternatively, we could drop these counties from the data.
Or, alternatively (again), we could impute values based on the average of their neighbors
- 12087: Monroe, FL; centroid is (-81.1 25.3) and neighbors are [12021, 12086]
- 25019: Nantucket, MA; centroid is (-70.1 41.3) but there are no neighbors
- 34017: Hudson, NJ; centroid is (-74.1 40.7) and neighbors are [36061, 34003, 34013]
- 37031: Carteret, NC; centroid is (-76.7 34.8) and neighbors are [37133, 37103, 37049]
- 48007: Aransas, TX; centroid is (-97 28.1) and neighbors are [48355, 48057, 48409, 48391]
- 51115: Mathews, VA; centroid is (-76.3 37.4) and neighbors are [51119, 51073]
- 51810: Virginia Beach, VA; centroid is (-76 36.7) and neighbors are [37053, 51710, 51550]
- 53029: Island, WA; centroid is (-122.5 48.2) and neighbors are [53061]

### Filling missing emPOWER data

We'll (re-)look up the emPOWER data for fips 46102 (which is listed as 46113 in the emPOWER data).

The (commented-out) code below was used to generate the following values. We'll fill these in manually
- 2023: 87.200000
- 2022: 79.083333
- 2021: 81.833333
- 2020: 84.666667
- 2019: 90.333333
- 2018: 92.666667
- 2017: 94.250000
- 2016: 98.750000
- 2015: 98.750000 (note that 2016 is the oldest data we have, so we'll assume the same values for 2015 and 2014)
- 2014: 98.750000

In [5]:
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2023), 'Power_Dependent_Devices_DME_Mean'] = 87.200000
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2022), 'Power_Dependent_Devices_DME_Mean'] = 79.083333
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2021), 'Power_Dependent_Devices_DME_Mean'] = 81.833333
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2020), 'Power_Dependent_Devices_DME_Mean'] = 84.666667
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2019), 'Power_Dependent_Devices_DME_Mean'] = 90.333333
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2018), 'Power_Dependent_Devices_DME_Mean'] = 92.666667
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2017), 'Power_Dependent_Devices_DME_Mean'] = 94.250000
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2016), 'Power_Dependent_Devices_DME_Mean'] = 98.750000
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2015), 'Power_Dependent_Devices_DME_Mean'] = 98.750000
df.loc[(df['fips_code'] == 46102) & (df['YEAR']==2014), 'Power_Dependent_Devices_DME_Mean'] = 98.750000

### Filling missing buried power line data

- 46102: Oglala Lakota, SD; this code transitioned from fips_code 46113, and should have a value of 0.1205137891106878
- 51720: Norton City, VA; this is in Wise County (fips_code 51195), and should have a value of 0.099491631

In [6]:
df.loc[df['fips_code'] == 46102, 'Pct_Buried_Lines'] = 0.1205137891106878
df.loc[df['fips_code'] == 51720, 'Pct_Buried_Lines'] = 0.099491631
df.loc[df['fips_code'] == 22101, 'Pct_Buried_Lines'] = 0.162259021

### Filling missing Subregion data

- 25001: Barnstable, MA is in the NEWE subregion
- 51131: Northampton, VA is in the RFCE subregion

In [7]:
df.loc[df['fips_code'] == 25001, 'Subregion'] = 'NEWE'
df.loc[df['fips_code'] == 51131, 'Subregion'] = 'RFCE'

## Feature Engineering

Some features have already been engineered as part of the process of merging the eaglei, NOAA, and county-level predictors

Below, we'll engineer features on the merged dataset. For each fips code, we'll compute
- The mean value of each ERA5 predictor from all the neighboring counties
- The max value of each ERA5 predictor from all neighboring counties
- The total 12-hour and 24-hour precipitation and snowfall within each county

Note that this takes ~23 minutes to run on my home computer

In [8]:
#Create a new dataframe
df_era5_grouped = pd.DataFrame()

#iterate through fips_code values
for fips_code in df['fips_code'].unique():

    #get the neighbors list for the current fips_code
    neighbors_list = df.loc[df['fips_code'] == fips_code, 'neighbors'].values[0]

    #make a new dataframe where fips_code is in neighbors_list
    df_neighbors = df[df['fips_code'].isin(neighbors_list)] 

    #group df_neighbors by datetime and compute the mean and max of t2m, u10, v10, sf, and tp
    df_neighbors_grouped = df_neighbors.groupby('datetime').agg({'t2m': ['mean', 'max'], 'u10': ['mean', 'max'], 'v10': ['mean', 'max'], 'sf': ['mean', 'max'], 'tp': ['mean', 'max']})

    #Make the grouped aggregated data into columns
    df_neighbors_grouped.columns = ['_'.join(col).strip() for col in df_neighbors_grouped.columns.values]

    #Add fips_code as a column
    df_neighbors_grouped['fips_code'] = fips_code

    #concatenate df_neighbors_grouped with df_era5_grouped
    df_era5_grouped = pd.concat([df_era5_grouped, df_neighbors_grouped], axis=0)

df_era5_grouped = df_era5_grouped.reset_index()

#Merge df_era5_grouped with df
df = df.merge(df_era5_grouped, on=['datetime', 'fips_code'], how='left')


In [ ]:
#Export df to a parquet file
df.to_parquet('../Data/Merged_Data/eaglei_noaa_era5_full_with_neighbors.parquet', index=False)

In [ ]:
##Old, non-vectorized code that takes forever to run...
##Compute the mean and maximum values of the ERA5 values of the neighbors for each row in df
#for i in df.index:
#    timeval = df.loc[i]['datetime']
#    neighbors_list = df.loc[i]['neighbors']
#    t2m_values = df.loc[(df['fips_code'].isin(neighbors_list)) & (df['datetime'] == timeval), 't2m']
#    u10_values = df.loc[(df['fips_code'].isin(neighbors_list)) & (df['datetime'] == timeval), 'u10']
#    v10_values = df.loc[(df['fips_code'].isin(neighbors_list)) & (df['datetime'] == timeval), 'v10']
#    sf_values = df.loc[(df['fips_code'].isin(neighbors_list)) & (df['datetime'] == timeval), 'sf']
#    tp_values = df.loc[(df['fips_code'].isin(neighbors_list)) & (df['datetime'] == timeval), 'tp']
#    df.loc[i, 't2m_neighbors_mean'] = t2m_values.mean()
#    df.loc[i, 't2m_neighbors_max'] = t2m_values.max()
#    df.loc[i, 'u10_neighbors_mean'] = u10_values.mean()
#    df.loc[i, 'u10_neighbors_max'] = u10_values.max()
#    df.loc[i, 'v10_neighbors_mean'] = v10_values.mean()
#    df.loc[i, 'v10_neighbors_max'] = v10_values.max()
#    df.loc[i, 'sf_neighbors_mean'] = sf_values.mean()
#    df.loc[i, 'sf_neighbors_max'] = sf_values.max()
#    df.loc[i, 'tp_neighbors_mean'] = tp_values.mean()
#    df.loc[i, 'tp_neighbors_max'] = tp_values.max()

### Imputing missing ERA5 data

Several ocean-adjacent counties are missing ERA5 data:
- 12087: Monroe, FL; centroid is (-81.1 25.3) and neighbors are [12021, 12086]
- 25019: Nantucket, MA; centroid is (-70.1 41.3) but there are no neighbors
- 34017: Hudson, NJ; centroid is (-74.1 40.7) and neighbors are [36061, 34003, 34013]
- 37031: Carteret, NC; centroid is (-76.7 34.8) and neighbors are [37133, 37103, 37049]
- 48007: Aransas, TX; centroid is (-97 28.1) and neighbors are [48355, 48057, 48409, 48391]
- 51115: Mathews, VA; centroid is (-76.3 37.4) and neighbors are [51119, 51073]
- 51810: Virginia Beach, VA; centroid is (-76 36.7) and neighbors are [37053, 51710, 51550]
- 53029: Island, WA; centroid is (-122.5 48.2) and neighbors are [53061]

We can now impute these using the mean of the values from their neighboring counties (except for Nantucket... which we should probably just drop)

In [10]:
#For the following values of fips_code, 
# replace NaN values of t2m, u10, v10, sf, and tp 
# with corresponding values of t2m_mean, u10_mean, v10_mean, sf_mean, and tp_mean
# 12087, 34017, 37031, 48007, 51115, 51810, and 53029
fips_codes = [12087, 34017, 37031, 48007, 51115, 51810, 53029]
for fips_code in fips_codes:
    df.loc[(df['fips_code'] == fips_code) & (df['t2m'].isna()), 't2m'] = df.loc[(df['fips_code'] == fips_code) & (df['t2m'].isna()), 't2m_mean']
    df.loc[(df['fips_code'] == fips_code) & (df['u10'].isna()), 'u10'] = df.loc[(df['fips_code'] == fips_code) & (df['u10'].isna()), 'u10_mean']
    df.loc[(df['fips_code'] == fips_code) & (df['v10'].isna()), 'v10'] = df.loc[(df['fips_code'] == fips_code) & (df['v10'].isna()), 'v10_mean']
    df.loc[(df['fips_code'] == fips_code) & (df['sf'].isna()), 'sf'] = df.loc[(df['fips_code'] == fips_code) & (df['sf'].isna()), 'sf_mean']
    df.loc[(df['fips_code'] == fips_code) & (df['tp'].isna()), 'tp'] = df.loc[(df['fips_code'] == fips_code) & (df['tp'].isna()), 'tp_mean']

### Computing weather features

- Wind speed (from u and v components)
- 12- and 24-hour total precipitation and snowfall

We could also compute duration of various events... need to decide if from NOAA or ERA5, or both...

In [11]:
#Create a new column 'wind_speed' which is the square root of the sum of the squares of the 'u' and 'v' columns
df['wind_speed'] = (df['u10']**2 + df['v10']**2)**0.5
df['neighbor_mean_wind_speed'] = (df['u10_mean']**2 + df['v10_mean']**2)**0.5
df['neighbor_max_wind_speed'] = (df['u10_max']**2 + df['v10_max']**2)**0.5

#Drop the 'u' and 'v' columns
df = df.drop(columns=['u10', 'v10', 'u10_mean', 'v10_mean', 'u10_max', 'v10_max'])

In [12]:
# Compute cumulative amounts of snowfall and precipitation over 12- and 24-hour intervals
df['sf_12h'] = df.groupby('fips_code')['sf'].shift(1) + df['sf']
df['sf_24h'] = df.groupby('fips_code')['sf'].shift(1) + df.groupby('fips_code')['sf'].shift(2) + df.groupby('fips_code')['sf'].shift(3) + df['sf']

df['tp_12h'] = df.groupby('fips_code')['tp'].shift(1) + df['tp']
df['tp_24h'] = df.groupby('fips_code')['tp'].shift(1) + df.groupby('fips_code')['tp'].shift(2) + df.groupby('fips_code')['tp'].shift(3) + df['tp']

In [13]:
#Export to a parquet file
df.to_parquet('../Data/Merged_Data/eaglei_noaa_era5_engineered.parquet', index=False)